# Annualizing Return and Volatility: the √T Rule and When It Fails

A daily volatility of 1% is reported as an annual volatility of about 16%, because $1\% \times \sqrt{252} = 15.9\%$. The square root
comes from one fact: the variance of a sum of independent returns is the sum of the variances. Means scale with $T$, standard deviations
with $\sqrt{T}$. When returns are autocorrelated, as they are for smoothed real-estate valuations, hedge funds with stale prices, or
momentum and mean-reverting markets, the rule breaks, and the annual volatility can be off by a factor of two. This notebook derives the
rule, tests it on simulated data, and shows how far it fails.

## 0. Math link

- [Variance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/05_variance.ipynb)
  and [standard deviation](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/06_standard_deviation.ipynb):
  $\operatorname{Var}(aX) = a^2 \operatorname{Var}(X)$, and the standard deviation carries the units of the data.
- [Sum of expectations](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/04_sum_of_expectations.ipynb)
  and [covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb):
  the variance of a sum is the sum of all variances and covariances.
- Log returns from [00/01](./01_simple_vs_log_returns.ipynb): they add over time, which makes the scaling exact.

## 1. Introduction

Risk reports mix horizons: daily VaR, monthly fund returns, annual volatility targets. To compare them, every number is converted to an
annual figure. The standard recipe multiplies means by the number of periods per year and volatilities by its square root. This is
correct for log returns that are independent from one period to the next, and wrong otherwise.

After this notebook you can:

- annualize mean, volatility and Sharpe ratio from daily, weekly or monthly data, stating the periodicity;
- derive the $\sqrt{T}$ rule from the variance of a sum;
- compute the correction for autocorrelated returns (the variance ratio);
- recognize smoothed return series and why their annualized volatility and Sharpe ratio are misleading;
- choose between 252, 260 and 365 periods per year and explain the difference.

## 2. Theory

### Sum of period returns

Let $\ell_1, \dots, \ell_T$ be log returns over $T$ consecutive periods, each with mean $\mu$ and variance $\sigma^2$. The $T$-period log
return is $L_T = \sum_{t=1}^T \ell_t$ (00/01). By linearity of expectation,

$$
E[L_T] = T\mu .
$$

For the variance, expand the square of the sum:

$$
\operatorname{Var}(L_T) = \sum_{t=1}^T \operatorname{Var}(\ell_t) + 2\sum_{s<t} \operatorname{Cov}(\ell_s, \ell_t) .
$$

### The √T rule

If the returns are uncorrelated, every covariance is zero, so $\operatorname{Var}(L_T) = T\sigma^2$ and

$$
\sigma_T = \sigma\sqrt{T} .
$$

With $T$ the number of periods per year (252 trading days, 52 weeks, 12 months):

$$
\mu_{\text{ann}} = T\mu, \qquad \sigma_{\text{ann}} = \sqrt{T}\,\sigma, \qquad SR_{\text{ann}} = \frac{T\mu}{\sqrt{T}\sigma} = \sqrt{T}\, SR .
$$

Only uncorrelatedness is needed, not normality: fat-tailed independent returns obey the same rule for the volatility.

### With autocorrelation

If returns are stationary with autocorrelation $\rho_k$ at lag $k$, the $T - k$ pairs at distance $k$ each contribute $\rho_k\sigma^2$:

$$
\operatorname{Var}(L_T) = T\sigma^2\Big[1 + 2\sum_{k=1}^{T-1}\Big(1 - \frac{k}{T}\Big)\rho_k\Big] = T\sigma^2 \cdot VR(T) .
$$

$VR(T)$ is the **variance ratio**. For an AR(1) process with $\rho_k = \rho^k$ and large $T$, $VR \to \frac{1 + \rho}{1 - \rho}$:

| $\rho$ | $-0.2$ | $-0.1$ | $0$ | $0.1$ | $0.3$ | $0.6$ |
|---|---|---|---|---|---|---|
| $VR(\infty)$ | 0.67 | 0.82 | 1 | 1.22 | 1.86 | 4.0 |
| volatility factor $\sqrt{VR}$ | 0.82 | 0.90 | 1 | 1.11 | 1.36 | 2.0 |

Positive autocorrelation (trends, stale prices) makes long-horizon risk larger than $\sqrt{T}$ suggests; negative autocorrelation (mean
reversion, bid–ask bounce) makes it smaller.

### Smoothed returns

Appraisal-based valuations (real estate, private equity) report a weighted average of the true return and last period's report:
$r^{\text{obs}}_t = (1 - \alpha) r_t + \alpha\, r^{\text{obs}}_{t-1}$. This is an AR(1) with $\rho = \alpha$. Its one-period variance is
$\frac{1 - \alpha}{1 + \alpha}\sigma^2$, smaller than the truth, while its long-run variance is the true one. Annualizing the reported
monthly volatility with $\sqrt{12}$ therefore understates the annual risk by the factor $\sqrt{(1-\alpha)/(1+\alpha)}$.

### Annualizing simple returns

For simple returns the exact annual return from an average period return is geometric: $(1 + \bar r_g)^T - 1$ with the geometric mean
$\bar r_g$. Multiplying the arithmetic mean of simple returns by $T$ is an approximation, good for small returns and short periods.

## 3. Worked Examples

### Example 1: a daily risk number

Daily log returns of a Swiss stock index: mean 0.03%, standard deviation 1.0%. Annual (252 days): mean $252 \times 0.03\% = 7.6\%$,
volatility $\sqrt{252} \times 1.0\% = 15.9\%$, Sharpe ratio (zero risk-free rate) $0.476$, against a daily Sharpe ratio of $0.03$.

### Example 2: from annual down to ten days

An annual volatility of 20% corresponds to a ten-trading-day volatility of $20\% \times \sqrt{10/252} = 3.98\%$. Regulatory ten-day VaR
is often obtained this way from one-day VaR: $\text{VaR}_{10} = \sqrt{10}\,\text{VaR}_1$.

### Example 3: a smoothed real-estate fund

A fund reports monthly returns with volatility 1.0%, so a naive annual volatility of $\sqrt{12} \times 1.0\% = 3.5\%$, and the reported
returns have a first-order autocorrelation of 0.6. If this comes from smoothing with $\alpha = 0.6$, the true monthly volatility is
$1.0\% \times \sqrt{1.6/0.4} = 2.0\%$ and the true annual volatility about 6.9%. The fund looks as safe as bonds while it is about as
risky as a balanced fund; its Sharpe ratio is overstated by the same factor of 2.

### Example 4: mean reversion

Daily returns with $\rho_1 = -0.1$ (and no further autocorrelation) have $VR(252) = 1 - 2 \times 0.1 \times 251/252 = 0.80$. The annual
volatility is $\sqrt{0.80} = 0.89$ times the $\sqrt{252}$-scaled daily volatility: 14.2% instead of 15.9%.

## 4. Finance Application

### Where it matters

- **Risk reporting.** Annual volatility, tracking error and Sharpe ratios in fund factsheets are almost always $\sqrt{T}$-scaled from
  daily or monthly data.
- **VaR and margin.** Banks scale one-day VaR to ten days with $\sqrt{10}$; clearing houses scale margin periods the same way.
- **Alternative investments.** Real estate, private equity, infrastructure and some hedge funds have smoothed returns; asset allocators
  unsmooth them (module 09) before putting them into an optimizer.
- **Comparing strategies.** A strategy measured on daily data and one on monthly data are only comparable after annualizing both, and
  only if the scaling is valid for both.

### The usual misreadings

- "Annual volatility is 252 times daily volatility." It is $\sqrt{252}$ times; 252 times the variance.
- "√T needs normal returns." It needs uncorrelated returns; normality is irrelevant for the volatility (but matters for scaling
  quantiles such as VaR).
- "This fund has a Sharpe ratio of 2 with 3% volatility." With smoothed returns, both numbers are artefacts of the reporting.
- "252 or 365 does not matter." It does if the daily data include weekends: mixing a 365-day calendar with trading-day returns
  inflates the annual volatility by $\sqrt{365/252} = 1.20$.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### Independent, fat-tailed daily returns

Forty years of daily log returns (252 trading days a year), drawn from a Student-t distribution with 4 degrees of freedom scaled to an
annual volatility of 16% and an annual mean of 6%. We measure the volatility of non-overlapping sums over $h$ days and compare it with
$\sqrt{h}$ times the daily volatility.

In [ ]:
periods_per_year = 252
years = 40
n_days = periods_per_year * years
mu_ann, vol_ann, nu = 0.06, 0.16, 4

z = rng.standard_t(nu, size=n_days) / np.sqrt(nu / (nu - 2))   # unit variance
daily = mu_ann / periods_per_year + vol_ann / np.sqrt(periods_per_year) * z

def horizon_vol(returns, h):
    n = len(returns) // h * h
    sums = returns[:n].reshape(-1, h).sum(axis=1)
    return sums.std(ddof=1), len(sums)

sigma_d = daily.std(ddof=1)
rows = []
for h in [1, 5, 21, 63, 126, 252]:
    vol_h, n_obs = horizon_vol(daily, h)
    rows.append({"horizon (days)": h, "observations": n_obs, "measured vol": vol_h,
                 "sqrt(h) x daily vol": np.sqrt(h) * sigma_d, "ratio": vol_h / (np.sqrt(h) * sigma_d)})
pd.DataFrame(rows).set_index("horizon (days)")

Measured volatilities agree with $\sqrt{h}$ scaling up to sampling noise, despite the fat tails. At the annual horizon only 40
observations remain, so the measured annual volatility has a standard error of about $1/\sqrt{2 \times 40} = 11\%$ of its value (06/04);
a ratio within roughly 0.8–1.2 is consistent with the rule.

In [ ]:
annual_mean = daily.mean() * periods_per_year
annual_vol = daily.std(ddof=1) * np.sqrt(periods_per_year)
print(f"annualized mean {annual_mean:.4f}, annualized vol {annual_vol:.4f}, "
      f"annualized Sharpe (rf = 0) {annual_mean / annual_vol:.3f}")
print(f"daily Sharpe {daily.mean() / daily.std(ddof=1):.4f} x sqrt(252) = "
      f"{daily.mean() / daily.std(ddof=1) * np.sqrt(periods_per_year):.3f}")

### Autocorrelated returns: AR(1)

The same innovations, filtered through $\ell_t = \rho\,\ell_{t-1} + \varepsilon_t$ and rescaled so that the daily volatility stays at
$16\%/\sqrt{252}$. The daily numbers look identical; the annual risk does not.

In [ ]:
def ar1(innovations, rho):
    x = np.empty_like(innovations)
    x[0] = innovations[0]
    for t in range(1, len(innovations)):
        x[t] = rho * x[t - 1] + innovations[t]
    return x / x.std() * vol_ann / np.sqrt(periods_per_year)     # same daily vol for every rho

def variance_ratio_ar1(rho, h):
    k = np.arange(1, h)
    return 1 + 2 * np.sum((1 - k / h) * rho**k)

rows = []
for rho in [-0.2, -0.1, 0.0, 0.1, 0.3]:
    x = ar1(z, rho)
    vol_1 = x.std(ddof=1) * np.sqrt(periods_per_year)
    vol_252, _ = horizon_vol(x, periods_per_year)
    vol_21, _ = horizon_vol(x, 21)
    rows.append({"rho": rho, "sqrt(252)-scaled daily vol": vol_1,
                 "from monthly sums, x sqrt(12)": vol_21 * np.sqrt(12),
                 "theory: sqrt(VR(252)) x scaled": vol_1 * np.sqrt(variance_ratio_ar1(rho, 252)),
                 "measured annual vol": vol_252})
pd.DataFrame(rows).set_index("rho")

Every row has the same daily volatility, 16% after scaling. With $\rho = 0.3$ the true annual volatility is about 22%; with
$\rho = -0.2$ it is about 13%. The measured annual numbers (40 observations each) scatter around the theory. Annualizing monthly sums
with $\sqrt{12}$ already captures most of the effect, because most of the autocorrelation is within a month.

### A smoothed real-estate fund

In [ ]:
months = 12 * 50
true_monthly = rng.normal(0.05 / 12, 0.07 / np.sqrt(12), size=months)   # true: 5% mean, 7% vol, annual
alpha = 0.6
reported = np.empty(months)
reported[0] = true_monthly[0]
for t in range(1, months):
    reported[t] = (1 - alpha) * true_monthly[t] + alpha * reported[t - 1]

def summary(r, name):
    ann_mean = r.mean() * 12
    ann_vol = r.std(ddof=1) * np.sqrt(12)
    annual_sums = r.reshape(-1, 12).sum(axis=1)
    return {"series": name, "annual mean": ann_mean, "monthly vol x sqrt(12)": ann_vol,
            "vol of annual sums": annual_sums.std(ddof=1), "lag-1 autocorr": np.corrcoef(r[1:], r[:-1])[0, 1],
            "Sharpe (rf = 0.5%)": (ann_mean - 0.005) / ann_vol}

pd.DataFrame([summary(true_monthly, "true"), summary(reported, "reported (smoothed)")]).set_index("series")

Fifty years of monthly data. The reported series has the same mean but a $\sqrt{12}$-scaled volatility of 3.4%, less than half the true
7.1%; its lag-1 autocorrelation is 0.55, close to $\alpha = 0.6$. The Sharpe ratio computed from the reported monthly data is about double the
true one. Measured on calendar-year sums, the two series have almost the same volatility (5.6% and 5.7%): smoothing only delays returns by a
few months. Both are below 7% because 50 annual observations are a noisy sample; the standard error is about 10% of the value.

In [ ]:
# Unsmoothing: invert the smoothing filter with the estimated autocorrelation
rho_hat = np.corrcoef(reported[1:], reported[:-1])[0, 1]
unsmoothed = (reported[1:] - rho_hat * reported[:-1]) / (1 - rho_hat)
pd.DataFrame([summary(unsmoothed[: (len(unsmoothed) // 12) * 12], "unsmoothed")]).set_index("series")

Inverting the filter with the estimated autocorrelation recovers most of the gap: 6.3% against the true 7.1%, and a Sharpe ratio of 0.83
against 0.73. The remaining shortfall comes from the estimated autocorrelation (0.55) being below the true 0.6. Module 09 treats
unsmoothing in more detail.

## 6. Visualizations

In [ ]:
hs = np.array([1, 2, 5, 10, 21, 42, 63, 126, 252])
fig, ax = plt.subplots(figsize=(10, 5))
for rho, color in [(-0.2, "C0"), (0.0, "C1"), (0.3, "C2")]:
    x = ar1(z, rho)
    measured = [horizon_vol(x, h)[0] for h in hs]
    ax.plot(hs, np.array(measured) * 100, "o", color=color, label=f"measured, rho = {rho}")
    ax.plot(hs, x.std(ddof=1) * np.sqrt(hs) * np.sqrt([variance_ratio_ar1(rho, h) for h in hs]) * 100, "-", color=color, lw=1)
ax.plot(hs, z.std() * vol_ann / np.sqrt(periods_per_year) * np.sqrt(hs) * 100, "k--", label="sqrt(h) rule")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_yticks([1, 2, 5, 10, 20])
ax.set_yticklabels(["1", "2", "5", "10", "20"])
ax.set_title("Volatility over h days: sqrt(h) rule against autocorrelated returns")
ax.set_xlabel("horizon h (trading days, log scale)")
ax.set_ylabel("volatility over the horizon (%, log scale)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

On log–log axes the $\sqrt{h}$ rule is a straight line of slope ½. Uncorrelated returns follow it. Positively autocorrelated returns
start on the line (same daily volatility) and bend above it within a few days; negatively autocorrelated returns bend below. The solid
lines, in the same colours, are the theoretical $\sqrt{h \cdot VR(h)}$ curves.

In [ ]:
rhos = np.linspace(-0.5, 0.7, 200)
fig, ax = plt.subplots(figsize=(9, 5))
for h in [5, 21, 252]:
    ax.plot(rhos, np.sqrt([variance_ratio_ar1(r_, h) for r_ in rhos]), label=f"h = {h} days")
ax.plot(rhos, np.sqrt((1 + rhos) / (1 - rhos)), "k--", label="limit sqrt((1 + rho) / (1 - rho))")
ax.axhline(1, color="grey", lw=0.8)
ax.set_title("Correction factor for the sqrt(h) rule, AR(1) returns")
ax.set_xlabel("lag-1 autocorrelation rho")
ax.set_ylabel("true vol / (sqrt(h) x one-period vol)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The correction factor depends on the autocorrelation much more than on the horizon once $h$ exceeds a few periods. Already
$\rho = 0.3$ means 36% more annual risk than the naive scaling; $\rho = 0.6$, typical of appraisal-based indices, doubles it.

In [ ]:
t = np.arange(months) / 12
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(t[:120], true_monthly[:120] * 100, lw=1, label="true")
axes[0].plot(t[:120], reported[:120] * 100, lw=2, label="reported (smoothed)")
axes[0].set_title("Monthly returns, first ten years")
axes[0].set_xlabel("years")
axes[0].set_ylabel("monthly return (%)")
axes[0].legend()

axes[1].plot(t, np.cumsum(true_monthly) * 100, label="true")
axes[1].plot(t, np.cumsum(reported) * 100, label="reported (smoothed)")
axes[1].set_title("Cumulative log return")
axes[1].set_xlabel("years")
axes[1].set_ylabel("cumulative log return (%)")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Month by month the reported series is much calmer. Over the long run the two cumulative paths stay close, with the smoothed one
lagging a few months behind: the risk is not removed, only delayed.

## 7. Exercises

### Basic

1. Weekly log returns have mean 0.12% and standard deviation 2.2%. Annualize the mean, the volatility and the Sharpe ratio (risk-free rate
   zero, 52 weeks a year).
2. A risk report gives a one-day 99% VaR of CHF 1.2 million. What is the $\sqrt{T}$-scaled ten-day VaR, and which assumption does it rely on?
3. A private real-estate fund reports an annualized volatility of 4% and a Sharpe ratio of 1.1 from quarterly returns whose lag-1
   autocorrelation is 0.5. **Interpret** these numbers for an investor who compares the fund with listed real estate (volatility around 15%).

### Intermediate

4. Compute $VR(h)$ exactly for an MA(1) process $\ell_t = \varepsilon_t + \theta\varepsilon_{t-1}$, which has $\rho_1 = \theta/(1 + \theta^2)$
   and $\rho_k = 0$ for $k \ge 2$. Check it against a simulation with $\theta = -0.3$ and $h = 21$.
5. Using the simulated fat-tailed daily returns, compare the 1% quantile of daily returns scaled by $\sqrt{10}$ with the 1% quantile of
   non-overlapping ten-day sums. Which is larger, and why?

### Advanced

6. Derive the variance-ratio formula $VR(T) = 1 + 2\sum_{k=1}^{T-1}(1 - k/T)\rho_k$ and its AR(1) limit $(1 + \rho)/(1 - \rho)$.
7. Show that for the smoothing model $r^{\text{obs}}_t = (1 - \alpha) r_t + \alpha r^{\text{obs}}_{t-1}$ with i.i.d. $r_t$, the variance of
   $r^{\text{obs}}_t$ is $\frac{1 - \alpha}{1 + \alpha}\sigma^2$ and its long-run variance $\lim_T \operatorname{Var}(\sum_t r^{\text{obs}}_t)/T$
   equals $\sigma^2$.

## 8. Solutions / Checks

In [ ]:
# 1.
m_w, s_w = 0.0012, 0.022
print(f"1. annual mean {52 * m_w:.4f}, annual vol {np.sqrt(52) * s_w:.4f}, Sharpe {52 * m_w / (np.sqrt(52) * s_w):.3f}")

# 2.
print(f"2. ten-day VaR {np.sqrt(10) * 1.2:.2f} CHF million")

# 3.
a3 = 0.5
factor = np.sqrt((1 + a3) / (1 - a3))
print(f"3. correction factor {factor:.2f}: volatility about {0.04 * factor:.1%}, Sharpe about {1.1 / factor:.2f}")

**2.** CHF 3.79 million, assuming daily P&L that is uncorrelated from day to day and a constant position and volatility over the ten days.
For normal returns the scaling of quantiles is then exact; for fat-tailed returns it is only approximate (exercise 5).

**3.** With a lag-1 autocorrelation of 0.5, smoothing understates the volatility by a factor of about $\sqrt{1.5/0.5} = 1.73$. The economic
volatility is closer to 7% and the Sharpe ratio closer to 0.6. The remaining gap to listed real estate (15%) reflects leverage, sector
mix and the daily repricing of listed shares; the fund is not three times safer, its reporting is three times slower.

In [ ]:
# 4. MA(1)
theta, h = -0.3, 21
rho1 = theta / (1 + theta**2)
vr_theory = 1 + 2 * (1 - 1 / h) * rho1
eps = rng.normal(size=200_000)
ma = eps[1:] + theta * eps[:-1]
n = len(ma) // h * h
vr_sim = ma[:n].reshape(-1, h).sum(axis=1).var() / (h * ma.var())
print(f"4. rho_1 = {rho1:.4f}, VR(21) theory {vr_theory:.4f}, simulated {vr_sim:.4f}")

# 5.
q_scaled = np.sqrt(10) * np.quantile(daily, 0.01)
ten_day = daily[: n_days // 10 * 10].reshape(-1, 10).sum(axis=1)
q_ten = np.quantile(ten_day, 0.01)
print(f"5. sqrt(10) x daily 1% quantile {q_scaled:.4f}, 1% quantile of ten-day sums {q_ten:.4f}")

**4.** For MA(1), only $\rho_1$ is non-zero, so $VR(h) = 1 + 2(1 - 1/h)\rho_1$. Simulation and formula agree.

**5.** The scaled daily quantile is further in the tail. The ten-day sum of fat-tailed returns is closer to normal (central limit theorem),
so its 1% quantile is smaller in units of its volatility than the daily one. Scaling a fat-tailed daily VaR by $\sqrt{10}$ is conservative here.
The variance still scales exactly; the quantile does not.

**6.** $\operatorname{Var}(\sum_t \ell_t) = \sum_{s,t}\operatorname{Cov}(\ell_s, \ell_t) = \sigma^2\sum_{s,t}\rho_{|s-t|}$. There are $T$ pairs with $s = t$
and $2(T - k)$ ordered pairs at distance $k$, so the sum is $\sigma^2\big[T + 2\sum_k (T - k)\rho_k\big]$; divide by $T\sigma^2$. For AR(1),
$\sum_{k\ge1}\rho^k = \rho/(1 - \rho)$ and the weights $1 - k/T \to 1$, so $VR \to 1 + 2\rho/(1 - \rho) = (1 + \rho)/(1 - \rho)$.

**7.** Stationarity gives $V = (1 - \alpha)^2\sigma^2 + \alpha^2 V$, so $V = \frac{(1 - \alpha)^2}{1 - \alpha^2}\sigma^2 = \frac{1 - \alpha}{1 + \alpha}\sigma^2$.
The series is AR(1) with $\rho = \alpha$, so the long-run variance is $V \cdot \frac{1 + \alpha}{1 - \alpha} = \sigma^2$. Alternatively, summing the
recursion over many periods shows that every true return eventually enters the reported sum with total weight
$(1 - \alpha)(1 + \alpha + \alpha^2 + \dots) = 1$.

## 9. Conclusion

- For uncorrelated log returns, means scale with $T$, volatilities with $\sqrt{T}$, Sharpe ratios with $\sqrt{T}$; normality is not needed.
- Autocorrelation multiplies the variance by $VR(T) = 1 + 2\sum_k (1 - k/T)\rho_k$; for AR(1) this tends to $(1 + \rho)/(1 - \rho)$.
- Smoothed series (appraisal-based assets) have positive autocorrelation and understated volatility and overstated Sharpe ratios.
- Quantiles such as VaR scale with $\sqrt{T}$ exactly only for normal returns; for fat tails the scaling is approximate.
- State the periodicity and the number of periods per year with every annualized figure.

This completes module 00. Next: [01/01 Bond price as a function of yield](../01_fixed_income_math/01_bond_price_as_a_function_of_yield.ipynb).